# Part 18 — Structured and SQL RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mftnakrsu/rag-by-hand/blob/main/part-18-structured-sql-rag/sql_rag.ipynb)

*Most enterprise knowledge lives in databases and tables, not documents. Retrieve the schema, generate SQL, execute, answer — and close the series.*

📖 Read the essay: https://www.mefby.com/essays/structured-sql-rag

🐍 Companion script: `sql_rag.py`

## What you'll build

For seventeen parts we treated knowledge as **text**: documents chunked into passages, embedded, retrieved by similarity, read by a model. That worked because the answer was *sitting* in a passage, waiting to be found. This final part confronts the knowledge that does not work that way: the rows, columns, and tables that hold most of what an enterprise actually knows.

Ask *"what was our total revenue from shipped orders?"* and there is **no passage** that holds the answer. The number does not exist as text until someone **computes** it: join orders to products, filter to shipped, multiply price by quantity, sum. Dense passage retrieval (the engine of every part before this one) finds text *similar in meaning* to the query. It is very good at that and completely unable to do arithmetic over a set of rows. So the pipeline changes shape. The document spine was *embed, retrieve, ground, generate*; the structured spine swaps two words:

- **retrieve schema** → pull the relevant *subset* of the database schema (not passages);
- **generate SQL** → write a query grounded in that schema (+ a glossary, + exemplars);
- **execute** → run the SQL against the database, which returns exact rows;
- **answer** → phrase the result rows as the answer.

Concretely you'll build, cell by cell:

- a tiny **SQLite** database (stdlib `sqlite3`) with three tables an e-commerce support bot might sit in front of;
- **schema cards** + a **domain glossary** (the unit you retrieve, and the map from business words to columns);
- `retrieve_schema(question, k)` — retrieve the relevant schema *subset* (**schema linking**), with a real embedder path behind `try/except` and a keyword-overlap fallback;
- `generate_sql(...)` — a transparent rule-based stub standing in for an LLM, that emits **real, executable** SQL;
- the `sql_rag()` loop that **executes** the SQL against real SQLite and answers from the rows;
- `route()` — the Part 15 callback that sends aggregational / numeric questions to SQL and everything else to the document path.

> **Runs with no network and no API key.** The demo is pure standard library (`sqlite3`). The **execution step is genuinely real**: the generated SQL runs against a real SQLite database. Two pieces are mocked so the file stays dependency-light — schema retrieval is keyword overlap (not embeddings) and SQL generation is a rule-based stub (not a model) — but the *control flow* is the production one exactly. The real schema-embedder path sits behind a `try/except`, so it lights up automatically when `sentence-transformers` is installed and a model is cached (still no network); only the scores change, the answers are identical either way.

## Setup

Two imports do the work: `re` for tokenizing the keyword scorer, and `sqlite3` from the standard library for the real database. Nothing here touches the network, and there is no model to download for the default path.

In [ ]:
import re
import sqlite3

print("stdlib sqlite3", sqlite3.sqlite_version, "— ready (no network, no API key required)")

## Step 1 — A tiny structured database

This is the kind of knowledge that lives in a **database**, not in documents: it is queried and aggregated, never "read". We build three tables an e-commerce support bot might sit in front of:

- **customers** — one row per customer, with a `tier` of `free` / `pro` / `enterprise`;
- **products** — one row per product, with a unit `price`;
- **orders** — one row per order, joining a customer to a product, with a `quantity` and a `status` (`shipped` / `refunded` / `pending`).

Notice that *revenue* is **not a column anywhere**. It has to be computed: `price * quantity`, summed. That absence is the whole reason this part exists. We seed a handful of rows so every answer is checkable by hand.

In [ ]:
def build_db():
    db = sqlite3.connect(":memory:")
    db.executescript(
        """
        CREATE TABLE customers (
            id      INTEGER PRIMARY KEY,
            name    TEXT,
            country TEXT,
            tier    TEXT          -- 'free' | 'pro' | 'enterprise'
        );
        CREATE TABLE products (
            id       INTEGER PRIMARY KEY,
            name     TEXT,
            category TEXT,
            price    REAL
        );
        CREATE TABLE orders (
            id          INTEGER PRIMARY KEY,
            customer_id INTEGER,
            product_id  INTEGER,
            quantity    INTEGER,
            status      TEXT,     -- 'shipped' | 'refunded' | 'pending'
            order_date  TEXT      -- ISO date
        );
        """
    )
    db.executemany(
        "INSERT INTO customers VALUES (?,?,?,?)",
        [
            (1, "Ada", "DE", "pro"),
            (2, "Bashir", "TR", "enterprise"),
            (3, "Chen", "SG", "free"),
            (4, "Dilan", "TR", "pro"),
        ],
    )
    db.executemany(
        "INSERT INTO products VALUES (?,?,?,?)",
        [
            (10, "Aurora Lamp", "lighting", 49.0),
            (11, "Nimbus Speaker", "audio", 129.0),
            (12, "Coil Cable", "audio", 9.0),
        ],
    )
    db.executemany(
        "INSERT INTO orders VALUES (?,?,?,?,?,?)",
        [
            (100, 1, 11, 2, "shipped", "2026-03-04"),
            (101, 2, 10, 1, "shipped", "2026-03-09"),
            (102, 2, 12, 5, "refunded", "2026-03-12"),
            (103, 4, 11, 1, "shipped", "2026-03-20"),
            (104, 3, 10, 3, "pending", "2026-03-28"),
        ],
    )
    db.commit()
    return db


db = build_db()
print("customers:", db.execute("SELECT COUNT(*) FROM customers").fetchone()[0])
print("products: ", db.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print("orders:   ", db.execute("SELECT COUNT(*) FROM orders").fetchone()[0])
print("note: there is NO 'revenue' column anywhere — it must be computed.")

## Step 2 — Schema cards and the domain glossary

A **schema card** is one short, retrievable description per table: its name, its columns, and a one-line gloss of what a row means. In production a card also carries column descriptions and a few sample rows, and you **embed** it. The card is the *unit you retrieve* in schema linking — the structured analog of a document chunk.

The **domain glossary** is the quiet hero. Users ask about "revenue", "active users", "churn" — business words that are not columns. The glossary maps each to the columns and computation that encode it: "revenue" is `price * quantity` summed over orders, **not** a `revenue` column that does not exist. Without it the model invents columns; with it the model writes the query you meant.

In [ ]:
SCHEMA_CARDS = {
    "customers": {
        "columns": ["id", "name", "country", "tier"],
        "doc": "one row per customer: their name, country code, and plan tier "
               "(free, pro, enterprise). Use for how many customers questions.",
    },
    "products": {
        "columns": ["id", "name", "category", "price"],
        "doc": "one row per product for sale: its name, category, and unit price. "
               "Revenue is computed from price here times order quantity.",
    },
    "orders": {
        "columns": ["id", "customer_id", "product_id", "quantity", "status",
                    "order_date"],
        "doc": "one row per order placed: which customer bought which product, "
               "the quantity, the status (shipped, refunded, pending), and the date. "
               "Revenue and refunded order counts come from here.",
    },
}

# Business word -> the columns/values that encode it. This is what stops a model
# guessing that "revenue" lives in a column that does not exist.
GLOSSARY = {
    "revenue": "price * quantity, summed over orders (no revenue COLUMN exists)",
    "enterprise customer": "customers.tier = 'enterprise'",
    "refund / refunded": "orders.status = 'refunded'",
}

for term, meaning in GLOSSARY.items():
    print(f"  {term:<22} = {meaning}")

## Step 3 — The text we score (or embed) per card

Before we can retrieve, we need the text that *represents* each card. We concatenate the table name, its column names, and the gloss — exactly what you would embed in production. The `_tokens` helper lowercases and splits into word/number tokens for the keyword scorer.

In [ ]:
def _tokens(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def _card_text(table, card):
    """The text we score / embed for one card: name + columns + the gloss."""
    return table + " " + " ".join(card["columns"]) + " " + card["doc"]


print(_card_text("products", SCHEMA_CARDS["products"]))

## Step 4 — The real schema-embedder path (headline; falls back transparently)

In production you **embed** each schema card and retrieve the nearest few by cosine — exactly as Part 6 embeds document chunks, now pointed at the schema. We try that first, behind a `try/except`. If `sentence-transformers` is missing or no model is cached, we drop to a keyword-overlap scorer (the offline default) and print a clear banner. **The real path is the headline; the lexical scorer keeps the cell executable with no model and no network.**

Either way, the retrieved *tables* are identical for these demo questions, so the lesson — retrieve a **subset** of the schema, never the whole catalog — is the same.

In [ ]:
def _load_real_embedder():
    try:
        from sentence_transformers import SentenceTransformer  # REAL path
        model = SentenceTransformer("all-MiniLM-L6-v2")
        print("[schema] using sentence-transformers to embed schema cards")
        return model
    except Exception as exc:  # not installed / no weights / offline
        print(f"[schema] sentence-transformers unavailable ({type(exc).__name__}); "
              "scoring schema cards by keyword overlap (offline default)")
        return None


_EMBEDDER = _load_real_embedder()


def _overlap_score(question, table, card):
    """Lexical fallback: count shared tokens between question and card text."""
    return len(_tokens(question) & _tokens(_card_text(table, card)))


def _embed_score(question, table, card):
    """Real path: cosine between the question and the card embedding."""
    import numpy as np
    vecs = _EMBEDDER.encode([question, _card_text(table, card)],
                            normalize_embeddings=True)
    return float(np.dot(vecs[0], vecs[1]))

## Step 5 — Schema retrieval: retrieve a subset, never the catalog

This is the move that makes text-to-SQL scale. We score every card against the question and keep the **top-k**. This is **schema linking**: instead of pasting the whole database (real schemas run to hundreds of tables and tens of thousands of tokens, and tabular reasoning degrades as the table set grows *well before* the context limit), you hand the model only the few tables the question needs.

`k=2` keeps the top two tables — enough for a join, small enough to keep the model focused. Watch what comes back for a revenue question: the `orders` and `products` cards, exactly the two the `SUM(price * quantity)` join needs.

In [ ]:
def retrieve_schema(question, k=2):
    """Score each table's card against the question, keep the top-k.
    Returns [(table, score, card)]. The point is that we retrieve a SUBSET of
    the schema, never the whole catalog; for a large database this schema-linking
    step is what makes text-to-SQL work at all."""
    scored = []
    for table, card in SCHEMA_CARDS.items():
        if _EMBEDDER is not None:
            score = _embed_score(question, table, card)
        else:
            score = _overlap_score(question, table, card)
        scored.append((table, score, card))
    scored.sort(key=lambda x: -x[1])
    return scored[:k]


def render_schema(retrieved):
    """The retrieved schema subset, formatted the way you'd paste it into a prompt."""
    lines = []
    for table, _score, card in retrieved:
        cols = ", ".join(card["columns"])
        lines.append(f"TABLE {table}({cols})  -- {card['doc']}")
    return "\n".join(lines)


retrieved = retrieve_schema("What was our total revenue from shipped orders?", k=2)
print("retrieved tables:", [t for t, _, _ in retrieved])
print()
print(render_schema(retrieved))

## Step 6 — SQL generation: a transparent rule-based stub

A real system sends *(question + retrieved schema + glossary + few-shot SQL exemplars)* to an LLM and parses the SQL it returns. Here we use a **rule-based stub** so the demo needs no model — but it emits **real, executable** SQL, and the control flow around it is identical to the production one.

Two things to read carefully:

- **Order matters.** The most specific intents (revenue, refunds) are matched *before* the generic customer count, so "how many orders were refunded?" is not swallowed by the "how many" branch.
- **The guard.** The revenue branch requires *both* `orders` and `products` in the retrieved schema. If retrieval handed us too few tables, the stub **declines** rather than guessing. On structured data, declining loudly beats fabricating a confident, ungrounded number — the production lesson of the whole part.

In [ ]:
def generate_sql(question, retrieved):
    q = question.lower()
    tables = {t for t, _, _ in retrieved}

    # "total revenue" -> revenue is price*quantity (glossary), so JOIN + SUM.
    # GUARD: needs BOTH orders and products in the retrieved schema; if retrieval
    # handed us too few tables, decline rather than guess (try k=1 to see this).
    if "revenue" in q and {"orders", "products"} <= tables:
        return (
            "SELECT SUM(p.price * o.quantity) "
            "FROM orders o JOIN products p ON o.product_id = p.id "
            "WHERE o.status = 'shipped';"
        )

    # "how many refunded orders" -> COUNT over orders filtered by status.
    if "refund" in q and "orders" in tables:
        return "SELECT COUNT(*) FROM orders WHERE status = 'refunded';"

    # "how many <tier> customers" -> COUNT over customers, filtered by tier.
    if "how many" in q and "customers" in tables:
        for tier in ("enterprise", "pro", "free"):
            if tier in q:
                return f"SELECT COUNT(*) FROM customers WHERE tier = '{tier}';"
        return "SELECT COUNT(*) FROM customers;"

    # No intent matched, or the retrieved schema is missing a needed table.
    # Surface the miss instead of guessing wrong SQL.
    return None


print(generate_sql("What was our total revenue from shipped orders?", retrieved))

## Step 7 — Execute and answer: the loop, end to end

Here is the step with **no analog in the document pipeline**, and the one that makes the answer trustworthy: we **execute** the generated SQL against the real SQLite database. The database computes the number exactly, so the model never has to. The result rows come back, and a final step phrases them as an answer (here, formatting the single scalar each demo query returns).

`sql_rag()` is the whole loop: retrieve schema → generate SQL → execute → answer. If generation declined (returned `None`), we say so cleanly rather than running anything.

In [ ]:
def answer_from_rows(question, sql, rows):
    """Turn result rows into a one-line answer. A real system passes rows back to
    the model to phrase; here we format the single scalar each demo query returns."""
    if not rows:
        return "No rows matched."
    return f"{rows[0][0]}"


def sql_rag(db, question, k=2):
    retrieved = retrieve_schema(question, k=k)
    sql = generate_sql(question, retrieved)
    if sql is None:
        return {
            "tables": [t for t, _, _ in retrieved],
            "sql": None,
            "answer": "I could not turn that into a query over the known schema.",
        }
    rows = db.execute(sql).fetchall()        # <-- the real execution step
    return {
        "tables": [t for t, _, _ in retrieved],
        "sql": sql,
        "answer": answer_from_rows(question, sql, rows),
    }


out = sql_rag(db, "What was our total revenue from shipped orders?")
print("tables:", out["tables"])
print("SQL:   ", out["sql"])
print("answer:", out["answer"], "  <- 129*2 + 49*1 + 129*1, only shipped orders")

## Step 8 — The router: text-search versus SQL (the Part 15 callback)

We now have two kinds of pipeline — the document path from Parts 6–17 and the SQL path from this one — and a real system needs to send each query to the right one. This is the **same** routing problem as Part 15, reusing the same machinery: a small, fast classifier reading cheap signals off the query.

Part 15 routed by *complexity* (none / single / multi). Here we add a second axis, *structure*. **Aggregational and numeric questions** ("how many", "count", "total", "sum", "average", "top", "revenue") route to **SQL** — these are precisely the questions no passage can answer. Everything else ("what is our refund policy?") routes to the **document path**.

The honest caveat is the dangerous direction is the silent one: a numeric question misrouted to documents will retrieve a plausible passage and let the model *guess* a number. So bias borderline numeric questions toward SQL, where a wrong query at least tends to fail loudly.

In [ ]:
SQL_SIGNALS = ("how many", "count", "total", "sum", "average", "avg",
               "most", "top", "revenue", "per ", "number of")


def route(question):
    q = question.lower()
    if any(s in q for s in SQL_SIGNALS):
        return "sql"
    return "documents"


for q in ["How many enterprise customers do we have?",
          "What is your refund policy?"]:
    print(f"  {route(q):<9} <- {q}")

## Step 9 — The demo: four questions, two paths

Now run four questions through the router. Three are aggregational and go to the SQL loop — retrieve a two-table schema subset, generate real SQL, execute it, answer. The fourth ("what is your refund policy?") is descriptive and falls to the document path, never touching the database (mocked here as a stub, since you already built the real one in Part 6).

The final line confirms every SQL answer matches the seeded data: `1` enterprise customer, `436.0` shipped revenue, `1` refunded order.

In [ ]:
DEMO = [
    "How many enterprise customers do we have?",
    "What was our total revenue from shipped orders?",
    "How many orders were refunded?",
    "What is your refund policy?",            # routes to the document path
]

for q in DEMO:
    r = route(q)
    print(f"Q: {q}")
    print(f"   route={r}")
    if r != "sql":
        print("   -> document path (Parts 6-17): retrieve passages, ground, generate.\n")
        continue
    out = sql_rag(db, q)
    print(f"   schema retrieved: {out['tables']}")
    print(f"   SQL: {out['sql']}")
    print(f"   answer: {out['answer']}\n")

checks = {
    "How many enterprise customers do we have?": "1",            # Bashir
    "What was our total revenue from shipped orders?": "436.0",  # 129*2 + 49*1 + 129*1
    "How many orders were refunded?": "1",                       # order 102
}
ok = all(sql_rag(db, q)["answer"] == want for q, want in checks.items())
print("all SQL answers match the seeded data:", ok)

## Step 10 — Break it on purpose: a schema-linking failure

The intuition lives where it breaks. `retrieve_schema` keeps the top-`k` tables. Drop `k` to **1** and re-run the revenue question. Now only `orders` is retrieved; the `products` table (which holds `price`) never makes it into the prompt, so the revenue guard fails and `generate_sql` correctly **declines** rather than guessing.

That is the right failure: the query needed two tables and only got one, so it refused instead of fabricating. The production lesson in one line: on structured data, retrieving too little schema is a *wrong (or absent)* answer, not a smaller one. **Most wrong answers on structured data are schema-linking failures**, not generation failures.

In [ ]:
out_k1 = sql_rag(db, "What was our total revenue from shipped orders?", k=1)
print("k=1 retrieved:", out_k1["tables"])
print("k=1 SQL:      ", out_k1["sql"])
print("k=1 answer:   ", out_k1["answer"])
print()
print("It declined instead of guessing a number — the right failure.")

## What you learned

- **Dense passage retrieval cannot answer a computed question.** A number like "total revenue" exists nowhere as text until a query computes it. Retrieving the nearest passage produces a *fabricated* number; the fix is a query, not a better embedder.
- **Text-to-SQL with RAG** swaps two words of the series spine: instead of *embed, retrieve, ground, generate* it is **retrieve schema, generate SQL, execute, answer**. The database does the arithmetic exactly, so the model never guesses it.
- **What you retrieve is more than table names**: schema cards with column descriptions, sample rows, and a **domain glossary** that maps business words ("revenue") to the columns and computation that encode them. The glossary and sample rows prevent most invented-column and wrong-format errors.
- **You retrieve a schema *subset* (schema linking)** rather than pasting the catalog, for two reasons: real schemas dwarf the prompt budget, and tabular reasoning degrades as the table set grows, well inside the context window.
- **Routing** sends aggregational / numeric questions to SQL and everything else to the document path — the **same** classifier as Part 15, with one more axis (structure on top of complexity). Bias borderline numeric questions toward SQL, because the misroute toward documents fails *silently* by fabricating a number.
- Everything ran **offline**: the execution step is real (SQL against real SQLite), while schema retrieval (keyword overlap, with a real-embedder path behind `try/except`) and SQL generation (a transparent rule stub) are mocked so the file is dependency-light. The control flow is the production one.

### The close of the series

That is the end of the road. Across eighteen parts you built RAG from first principles: why it exists, how embeddings turn meaning into geometry, how to retrieve and rerank and ground and generate, how to measure it, how to run it in production, and now how to reach the structured knowledge that lives in databases rather than documents. The throughline never changed: **retrieve the relevant thing, then reason over it, and add complexity only where the evidence demands it.** Whether the relevant thing is a passage or a schema, the discipline is the same. The production capstone still lives in **Part 12**. Now go point all of it at a real corpus, and a real database, and build something that does not make things up.